# 微調 Encoder 模型：中文情緒分類

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/03-Finetune-Text-Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

建議環境：Google Colab，`執行階段 → 變更執行階段類型 → T4 GPU`

In [ ]:
!pip install -q -U transformers
# 下載本章練習的參考解答 (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['q_errors.py', 'q_other_model.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline
device = "cuda" if torch.cuda.is_available() else "cpu"

## 資料集：ChnSentiCorp

中文評論（飯店、書籍、電子產品）情緒分類，`label` 1 是正面、0 是負面。

`datasets` 讀進來的是 Apache Arrow 格式，不會一次全部載入記憶體，可以處理很大的資料集。

In [ ]:
ds = load_dataset("lansinuote/ChnSentiCorp")
ds = ds.filter(lambda x: x["text"] is not None)
ds

In [ ]:
ds["train"][0]

In [ ]:
ds["train"].to_pandas().head(10)

In [ ]:
df = ds["train"].to_pandas()
print(df.label.value_counts())
df.text.str.len().describe()

## 模型：mmBERT

Encoder 模型（BERT 家族）很適合分類、檢索等「理解」型的任務，比 LLM 小很多也快很多。

[mmBERT](https://huggingface.co/jhu-clsp/mmBERT-small) (2025) 是基於 ModernBERT 架構的多語言模型，支援 1800 多種語言，最長 8192 tokens。

In [ ]:
model_id = "jhu-clsp/mmBERT-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer("這家飯店的服務很好")

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)
ds_tok = ds.map(tokenize, batched=True, remove_columns=["text"])
ds_tok

`AutoModelForSequenceClassification` 會在預訓練模型上加一個隨機初始化的分類層。
（會看到一個警告說有些權重是新初始化的，這是正常的）

In [ ]:
id2label = {0: "負面", 1: "正面"}
label2id = {v: k for k, v in id2label.items()}
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2, id2label=id2label, label2id=label2id)
print(f"{model.num_parameters()/1e6:.0f}M 參數")

## 訓練前：模型在做什麼？

輸入 token ids，輸出每個類別的 logits。現在分類層是隨機的，所以預測也是隨便猜。

In [ ]:
model.to(device)
inputs = tokenizer(["這家飯店的服務很好", "房間又小又髒"], padding=True, return_tensors="pt").to(device)
with torch.no_grad():
    print(model(**inputs).logits.softmax(-1))

## 使用 `Trainer` 微調

`Trainer` 幫我們處理訓練迴圈、混合精度、評估、存檔、logging 等。
* `fp16=True`：T4 上用 fp16 混合精度
* `processing_class=tokenizer`：會自動用 `DataCollatorWithPadding` 把每個 batch padding 成一樣長

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": (logits.argmax(-1) == labels).mean()}

args = TrainingArguments(
    output_dir="chnsenticorp-mmbert",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    num_train_epochs=2,
    warmup_steps=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
trainer.evaluate(ds_tok["test"])

## 存檔與使用

存成資料夾後，就能像 Hub 上的模型一樣用 `from_pretrained` 或 `pipeline` 讀取。
也可以用 `trainer.push_to_hub()` 上傳到 Hugging Face Hub（需要登入）。

In [ ]:
trainer.save_model("my-sentiment-model")
clf = pipeline("text-classification", model="my-sentiment-model", device=device)
clf(["這家飯店的服務很好，早餐也很豐盛", "房間又小又髒，隔音很差", "價格便宜，但是位置有點偏僻"])

## Q
找出 test set 中模型預測錯誤的例子，看看是什麼樣的評論比較困難？

提示：`trainer.predict(ds_tok["test"])`

In [ ]:
# %load q_errors.py
%run -i q_errors.py

## Q
換成其他模型比較看看，例如
* `google-bert/bert-base-chinese`（經典的中文 BERT，2018）
* `jhu-clsp/mmBERT-base`（比較大的 mmBERT）

比較準確率和訓練時間。

In [ ]:
# %load q_other_model.py
%run -i q_other_model.py